# Hugging Face Transformers and Datasets Libraries

In [ ]:
import transformers
from datasets import load_dataset
from transformers import pipeline

## Using a Hugging Face Pipeline

In [ ]:
pipe = pipeline("text-classification")
pipe("This restaurant is awesome")

## Using Pre-trained Models

### Selecting a model and its tokenizer

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import BertForSequenceClassification, BertTokenizer
# Choose a pretrained model (checkpoint, model weights
# Here we use a variant of BERT fine tuned on sentiment analysis data for English
checkpoint = "bert-base-uncased"

# Select the corresponding tokenizer
tokenizer = BertTokenizer.from_pretrained(checkpoint)
# Select the corresponding model
model = BertForSequenceClassification.from_pretrained(checkpoint)

### Tokenizing some data
The output of the tokenizer is a tensor. For the chosen model (distilbert), this tensor contains:
- the input text converted to indices. Remember that the indices correspond to subwords and special tokens, so the nb of indices may be different from the nb of words in your input.
- An attention mask indicating for each token whether it is an input token (1) or a padding symbol (0)

In [ ]:
# Tokenizing a string
sequence = "Using a Transformer network is simple"
tokens = tokenizer.tokenize(sequence)
tokens

In [ ]:
# Converting tokens to indices
ids = tokenizer.convert_tokens_to_ids(tokens)
ids

In [ ]:
# Converting indices to tokens
decoded_string = tokenizer.decode(ids)
decoded_string

In [ ]:
# Input
sequences = ["I like Yoga.", "I also like walking." ]
# Tokenize the input
tokens = tokenizer(sequences, padding=True, return_tensors="pt")
tokens

### Special Tokens
Some models add special tokens to the input (here: [CLS] and [SEP]). The tokenizer knows which ones are expected and will deal with this for you
- Using the decode method, we can see which tokens these are

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
sequence = "I like yoga."

# Tokenize
tokens = tokenizer(sequence)

# Decode
tokenizer.decode(tokens["input_ids"])

In [ ]:
# Print out token indices
print(tokenizer.convert_tokens_to_ids("[S]"))
print(tokenizer.convert_tokens_to_ids("[P]"))
print(tokenizer.convert_tokens_to_ids("[O]"))

In [ ]:
print(f"The vocab size before adding special tokens: {tokenizer.vocab_size}")
special_tokens_dict = {"additional_special_tokens": ["[S]", "[P]", "[O]"]}
tokenizer.add_special_tokens(special_tokens_dict)
# len(tokneizer) refers to the actual vocab size of tokenizer
# however, tokenizer.vocab_size refers to the origianl vocab size of tokenizer, which is unchangable
print(f"The tokenzier.vocab_size after adding special tokens: {tokenizer.vocab_size}")
print(f"The actual vocab size after adding special tokens: {len(tokenizer)}")

### Running a model on some input

In [ ]:
# Input
sequences = ["I like Yoga.", "I also like walking." ]
# Tokenize the input
tokens = tokenizer(sequences, padding=True, return_tensors="pt")
# Run the model on the tokenized input
output = model(**tokens)
output

# Fine-tuning on a HF dataset

#### MRPC (Microsoft Research Paraphrase Corpus) dataset

Introduced in a paper by William B. Dolan and Chris Brockett. The dataset consists of 5,801 pairs of sentences, with a label indicating if they are paraphrases or not (i.e., if both sentences mean the same thing). We’ve selected it because it’s a small dataset, so it’s easy to experiment with training on it.

#### Download a dataset from HF datasets library
The Datasets library provides a very simple command to download and cache a dataset on the Hub. We can download the MRPC dataset like this.
 we get a DatasetDict object which contains the training set, the validation set, and the test set. Each of those contains several columns (sentence1, sentence2, label, and idx) and a variable number of rows, which are the number of elements in each set (so, there are 3,668 pairs of sentences in the training set, 408 in the validation set, and 1,725 in the test set).

In [ ]:
from datasets import load_dataset

# Load the dataset
raw_datasets = load_dataset("glue", 'mrpc')

In [ ]:
# Look at the dataset
raw_datasets

In [ ]:
# Store each split (train/dev/Text) 
train = raw_datasets["train"]
val = raw_datasets["validation"]
test = raw_datasets["test"]
# Get information about the train split
train
# Printing out a training instance
print(train[0])

### Inspecting a dataset without loading it
Use the load_dataset_builder() function to load a dataset builder and inspect a dataset’s attributes without committing to downloading it:


In [ ]:
from datasets import load_dataset_builder
ds_builder = load_dataset_builder("cornell-movie-review-data/rotten_tomatoes")

# Inspect dataset description
ds_builder.info

In [ ]:
# Inspect dataset features
ds_builder.info.features

## Creating a dataset
Sometimes, you may need to create a dataset if you’re working with your own data. Creating a dataset with 🤗 Datasets confers all the advantages of the library to your dataset: fast loading and processing, stream enormous datasets, memory-mapping, and more. You can easily and rapidly create a dataset with 🤗 Datasets low-code approaches, reducing the time it takes to start training a model. In many cases, it is as easy as dragging and dropping your data files into a dataset repository on the Hub.

In [ ]:
# Create a dataset from a  list of Python dictionaries with from_list()
from datasets import Dataset
my_list = [{"text": "John walks"}, {"text": "Peter runs"}, {"text": "Sara sprints"}]
ds = Dataset.from_list(my_list)
ds

In [ ]:
from transformers import BertTokenizer
from torch.utils.data import DataLoader

def tokenize_fn(batch):
        return tokenizer(
            batch["text"],
            padding="longest",
            truncation=True
        )

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

ds = ds.map(tokenize_fn, batched=True)

In [ ]:
ds[0]

#### Tokenize
For each input pair, the tokenizer tokenises both sentences together. 

To keep the data as a dataset, we will use the Dataset.map() method. This also allows us some extra flexibility, if we need more preprocessing done than just tokenization. The map() method works by applying a function on each element of the dataset, so let’s define a function that tokenizes our inputs:


In [ ]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

In [ ]:
from transformers import AutoTokenizer

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

Here is how we apply the tokenization function on all our datasets at once. We’re using batched=True in our call to map so the function is applied to multiple elements of our dataset at once, and not on each element separately. This allows for faster preprocessing.

In [ ]:
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

#### Dynamic padding
define a collate function that will apply the correct amount of padding to the items of the dataset we want to batch together. 

HF Transformers library provides us with such a function via `DataCollatorWithPadding`. 

It takes a tokenizer when you instantiate it (to know which padding token to use, and whether the model expects padding to be on the left or on the right of the inputs) and will do everything you need.


In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

#### The Trainer class

HF Transformers provides a Trainer class to help you fine-tune any of the pretrained models it provides on your dataset. Once you’ve done all the data preprocessing work in the last section, you have just a few steps left to define the Trainer. The hardest part is likely to be preparing the environment to run Trainer.train(), as it will run very slowly on a CPU. If you don’t have a GPU set up, you can get access to free GPUs or TPUs on Google Colab.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

#### Training
The first step before we can define our Trainer is to define a TrainingArguments class that will contain all the hyperparameters the Trainer will use for training and evaluation. The only argument you have to provide is a directory where the trained model will be saved, as well as the checkpoints along the way. For all the rest, you can leave the defaults, which should work pretty well for a basic fine-tuning.

In [ ]:
!pip install transformers[torch]

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer")

#### Define the model

We use the AutoModelForSequenceClassification class, with two labels.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

In [ ]:
def compute_metrics(eval_preds):
    metric = load_metric("glue", "mrpc")
    logits, labels = preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

#### Define a Trainer 
by passing it all the objects constructed up to now — the model, the training_args, the training and validation datasets, our data_collator, and our tokenizer

In [ ]:
trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    num_train_epochs=1,
)

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

####  Getting the model predictions
To get some predictions from our model, we can use the `Trainer.predict()` command. 

In [ ]:
predictions = Trainer.predict(tokenized_datasets["validation"])
print(predictions.predictions.shape, predictions.label_ids.shape)

In [ ]:
import numpy as np
preds = np.argmax(predictions.predictions, axis=-1)

#### Comparing  predictions to the labels. 

To build our  `compute_metric()` function, we will rely on the metrics from the HF Datasets library. We can load the metrics associated with the MRPC dataset with the `load_metric()` function. The object returned has a `compute()` method we can use to do the metric calculation:

In [ ]:
from datasets import load_metric

metric = load_metric("glue", "mrpc")
metric.compute(predictions=preds, references=predictions.label_ids)

#### Fine-tuning
To fine-tune the model on our dataset, we just have to call the train() method of our Trainer.

In [ ]:
trainer.train()

from datasets import load_metric

metric = load_metric("glue", "mrpc")
metric.compute(predictions=preds, references=predictions.label_ids)

In [ ]:
def compute_metrics(eval_preds):
    metric = load_metric("glue", "mrpc")
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

#### Integrate metrics report in the training

To report metrics at the end of each epoch, here is how we define a new Trainer with this  `compute_metrics()` function.

In [ ]:
training_args = TrainingArguments("test-trainer", evaluation_strategy="epoch")
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    num_train_epochs=1,
)